# 06 · A/B Test Simulation

Digital Onboarding Optimization Case Study

> Fully synthetic dataset. No real company, customer, or proprietary information is included.

## 1. Objective

This notebook designs and simulates an A/B test for a redesigned document-submission screen.

The proposed experiment tests whether a clearer, more persuasive, and less deferrable document screen can increase the `upload_now` rate and improve downstream approval conversion.

### Experiment idea

- **Control:** current screen with a visible option to upload later.
- **Treatment:** redesigned screen with stronger copy, better visual hierarchy, and a clearer recommendation to complete identity verification immediately.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest

pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data"
IMAGES_DIR = BASE_DIR / "images"
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

users = pd.read_csv(DATA_DIR / "onboarding_users.csv", parse_dates=["signup_timestamp"])
events = pd.read_csv(DATA_DIR / "events.csv", parse_dates=["event_timestamp"])
support = pd.read_csv(DATA_DIR / "support_calls.csv", parse_dates=["call_timestamp"])

users["has_support_call"] = users["user_id"].isin(support["user_id"]).astype(int)
users["upload_now"] = users["upload_choice"].eq("upload_now").astype(int)

users.head()

## 2. Baseline Metrics

In [ ]:
baseline = pd.Series({
    "users": len(users),
    "upload_now_rate": users["upload_now"].mean(),
    "document_submission_rate": users["document_submitted"].mean(),
    "approval_rate": users["approved"].mean(),
    "support_contact_rate": users["has_support_call"].mean()
}).to_frame("baseline_value")

baseline

## 3. Minimum Detectable Effect

In [ ]:
baseline_rate = users["upload_now"].mean()
target_lift = 0.12
target_rate = baseline_rate * (1 + target_lift)

baseline_rate, target_rate

In [ ]:
def required_sample_size_two_proportions(p1, p2, alpha=0.05, power=0.80):
    z_alpha = stats.norm.ppf(1 - alpha / 2)
    z_beta = stats.norm.ppf(power)
    pooled = (p1 + p2) / 2
    numerator = (z_alpha * np.sqrt(2 * pooled * (1 - pooled)) + z_beta * np.sqrt(p1 * (1 - p1) + p2 * (1 - p2))) ** 2
    denominator = (p2 - p1) ** 2
    return int(np.ceil(numerator / denominator))

sample_per_group = required_sample_size_two_proportions(baseline_rate, target_rate)
pd.DataFrame({
    "baseline_upload_now_rate": [baseline_rate],
    "target_upload_now_rate": [target_rate],
    "relative_lift": [target_lift],
    "required_sample_per_group": [sample_per_group],
    "total_required_sample": [sample_per_group * 2]
})

## 4. Simulating the Experiment

In [ ]:
rng = np.random.default_rng(2026)

experiment_n = min(sample_per_group * 2, len(users))
experiment = users.sample(experiment_n, random_state=2026).copy()
experiment["variant"] = rng.choice(["control", "treatment"], size=len(experiment), p=[0.5, 0.5])

# Treatment effect assumptions:
# 1. Higher probability of upload_now.
# 2. Slight improvement in document submission due to reduced ambiguity.
# 3. Lower support contact rate due to clearer guidance.

base_upload_prob = experiment["upload_now"].astype(float).values
treatment_boost = 0.11

sim_upload_now = []
for is_treatment, original in zip(experiment["variant"].eq("treatment"), experiment["upload_now"]):
    if is_treatment:
        if original == 1:
            sim_upload_now.append(1)
        else:
            sim_upload_now.append(int(rng.random() < treatment_boost))
    else:
        sim_upload_now.append(int(original))

experiment["sim_upload_now"] = sim_upload_now

# Simulate approval using observed conditional conversion rates
approval_now = users.loc[users["upload_now"] == 1, "approved"].mean()
approval_later = users.loc[users["upload_now"] == 0, "approved"].mean()

experiment["sim_approved"] = np.where(
    experiment["sim_upload_now"].eq(1),
    rng.binomial(1, approval_now, size=len(experiment)),
    rng.binomial(1, approval_later, size=len(experiment))
)

# Simulate support reduction for treatment users
support_now = users.loc[users["upload_now"] == 1, "has_support_call"].mean()
support_later = users.loc[users["upload_now"] == 0, "has_support_call"].mean()

experiment["sim_support_call"] = np.where(
    experiment["sim_upload_now"].eq(1),
    rng.binomial(1, support_now, size=len(experiment)),
    rng.binomial(1, support_later, size=len(experiment))
)

experiment.head()

## 5. Experiment Results

In [ ]:
ab_results = experiment.groupby("variant").agg(
    users=("user_id", "count"),
    upload_now_rate=("sim_upload_now", "mean"),
    approval_rate=("sim_approved", "mean"),
    support_contact_rate=("sim_support_call", "mean")
)

ab_results

In [ ]:
metrics = ["upload_now_rate", "approval_rate", "support_contact_rate"]
ax = ab_results[metrics].T.plot(kind="bar", figsize=(10, 5))
ax.set_title("A/B Test Results by Variant")
ax.set_ylabel("Rate")
ax.set_xlabel("Metric")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 6. Statistical Test

In [ ]:
control = experiment[experiment["variant"] == "control"]
treatment = experiment[experiment["variant"] == "treatment"]

def prop_test(metric):
    count = np.array([treatment[metric].sum(), control[metric].sum()])
    nobs = np.array([len(treatment), len(control)])
    z, p = proportions_ztest(count, nobs)
    return pd.Series({
        "control_rate": control[metric].mean(),
        "treatment_rate": treatment[metric].mean(),
        "absolute_lift": treatment[metric].mean() - control[metric].mean(),
        "relative_lift": treatment[metric].mean() / control[metric].mean() - 1,
        "z_statistic": z,
        "p_value": p
    })

test_results = pd.DataFrame({
    "upload_now": prop_test("sim_upload_now"),
    "approval": prop_test("sim_approved"),
    "support_contact": prop_test("sim_support_call")
}).T

test_results

## 7. Executive Interpretation

The simulated experiment suggests that a redesigned document-submission screen could meaningfully increase immediate upload behavior and approval conversion.

The next step is to connect these changes to business value:

- Incremental approved users
- Reduced support demand
- Operational savings
- Potential revenue impact